# SAR SICD experiments — inverse ω-K and inverse PFA

Run this top to bottom in Google Colab. It will:

1. install the libraries
2. upload the scripts from your machine
3. **self-test with synthetic data** (no download needed — do this first)
4. find and download a real SICD from Capella's free open-data bucket
5. run the inverse ω-K on it (SICD → raw data) and plot the result
6. optionally run the inverse PFA (SICD → CPHD) and the real-vs-synthetic test

Runtime → *Change runtime type* → CPU is fine. No AWS account is needed; the
open-data buckets are public.

## 1. Install

In [ ]:
!pip -q install "sarpy==2.1.1" finufft boto3 lxml shapely matplotlib numpy scipy
import numpy, sarpy, finufft, boto3
print("numpy", numpy.__version__, "| sarpy", sarpy.__version__, "| finufft", finufft.__version__)

## 2. Upload the scripts

Run the cell and pick `inverse_omega_k.py` (and `sicd_to_cphd.py` if you want
section 6). Colab clears these when the runtime restarts, so re-run then.

In [ ]:
from google.colab import files
import os
up = files.upload()          # choose inverse_omega_k.py [, sicd_to_cphd.py]
print()
for name in up:
    print(f"{name}: {os.path.getsize(name)/1024:.0f} KB")

## 3. Self-test on synthetic data

No input file. The script builds point-target raw data from the analytic radar
model, focuses it with the forward ω-K blocks, inverts it back, and scores every
step. **All five checks should say `ok` and it should print `SELF TEST PASSED`.**
If this fails, stop here — nothing downstream will be trustworthy.

In [ ]:
!python inverse_omega_k.py --self-test --quicklook -o /content/out/raw.npz

In [ ]:
from IPython.display import Image, display
display(Image("/content/out/self_test.png"))
# left: focused image from the synthetic raw | middle: raw data regenerated by the
# inverse blocks (targets smeared across azimuth) | right: refocused again

## 4. Find a real SICD in Capella's open data

Public bucket `capella-open-data`, no credentials. Filenames encode the mode:
`SP` spotlight, `SM` stripmap, `SS` sliding spotlight.

**ω-K assumes stripmap**, so prefer `SM` for section 5. For section 6 (inverse
PFA → CPHD) prefer `SP`/`SS`.

Listing is paginated and the bucket is large, so we scan one month prefix and
stop once we have enough candidates. Change `PREFIX` to look elsewhere.

In [ ]:
import boto3
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "capella-open-data"
PREFIX = "data/2024/2/"        # year/month; widen to "data/2024/" to scan more
WANT   = "_SM_SICD_"           # "_SM_SICD_" stripmap | "_SP_SICD_" spotlight | "_CPHD_"
MAX_HITS = 15

s3 = boto3.client("s3", region_name="us-west-2", config=Config(signature_version=UNSIGNED))
hits = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=PREFIX):
    for o in page.get("Contents", []):
        k = o["Key"]
        if WANT in k and k.endswith((".ntf", ".nitf", ".cphd")):
            hits.append((o["Size"], k))
    if len(hits) >= MAX_HITS:
        break

hits.sort()
print(f"{len(hits)} match {WANT!r} under {PREFIX!r} (smallest first):\n")
for i, (sz, k) in enumerate(hits[:MAX_HITS]):
    print(f"[{i:2d}] {sz/1e6:8.1f} MB  {k}")
if not hits:
    print("nothing found — try a different PREFIX (e.g. 'data/2023/') or WANT value")

## 5. Download one and run the inverse ω-K

Pick an index from the list above. Start small — a few hundred MB — and keep the
chip modest; Colab's free tier has limited RAM.

In [ ]:
import os, boto3
from botocore import UNSIGNED
from botocore.config import Config

CHOICE = 0                       # index from the table above
size, key = hits[CHOICE]
local = "/content/" + key.rsplit("/", 1)[-1]
if not os.path.exists(local):
    print(f"downloading {size/1e6:.1f} MB ...")
    boto3.client("s3", region_name="us-west-2",
                 config=Config(signature_version=UNSIGNED)).download_file(BUCKET, key, local)
print("ready:", local, f"{os.path.getsize(local)/1e6:.1f} MB")

In [ ]:
# chip-size 512 is a good start; raise it once you know the memory headroom.
# --roundtrip refocuses the generated raw data and scores it against the input.
!python inverse_omega_k.py "{local}" --chip-size 512 -o /content/out/raw.npz --roundtrip --quicklook

In [ ]:
from IPython.display import Image, display
display(Image("/content/out/raw.png"))

In [ ]:
import numpy as np
d = np.load("/content/out/raw.npz")
raw = d["raw"]
print("raw data:", raw.shape, raw.dtype)
print({k: float(d[k]) for k in ("f0","fs","bandwidth","Kr","velocity","r_ref","azimuth_spacing","prf")})

# range-compress the generated raw data: targets should appear as curved traces
# (range migration), which is the thing the image domain has already removed.
C = 299792458.0
f_tau = np.fft.fftfreq(raw.shape[0], d=1/float(d["fs"]))
rc = np.fft.ifft(np.fft.fft(raw, axis=0) * np.exp(1j*np.pi*f_tau**2/float(d["Kr"]))[:, None], axis=0)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
db = lambda x: np.clip(20*np.log10(np.abs(x)+1e-12) - 20*np.log10(np.abs(x).max()), -45, 0)
ax[0].imshow(db(raw), cmap="viridis", aspect="auto"); ax[0].set_title("raw (unfocused)")
ax[1].imshow(db(rc),  cmap="viridis", aspect="auto"); ax[1].set_title("range-compressed: migration traces")
for a in ax: a.set_xlabel("azimuth sample"); a.set_ylabel("range sample")
plt.tight_layout(); plt.show()

## 6. Optional — inverse PFA: SICD → CPHD, and the real-vs-synthetic test

This needs `sicd_to_cphd.py` uploaded in section 2. It works on **spotlight**
SICDs (PFA-formed) and refuses anything else, which is deliberate.

The interesting run is with a matching real CPHD, because then it runs:

* **Test A** — image the real CPHD onto the SICD's own grid and compare. This is
  the geometry go/no-go; a mirrored or shifted match means the conventions are wrong.
* **Test B** — compare synthetic and real phase history pulse by pulse. The phase
  curve it recovers is essentially the autofocus baked into the SICD.

In [ ]:
# Umbra's bucket publishes SICD and CPHD for the same collect, which is what the
# comparison needs. This lists candidate pairs (first call walks the bucket, ~20 s).
!python sicd_to_cphd.py --list-pairs 10 --download-dir /content/umbra

In [ ]:
# Download the smallest pair, convert, and run both tests.
# Add --task-filter "Beet Piler" to pick a specific site. Mind the disk quota:
# a SICD + CPHD pair is often ~800 MB.
!python sicd_to_cphd.py --download auto --download-dir /content/umbra \
    -o /content/cphd_out --chip-size 2048 --compare-chip 1024

In [ ]:
import glob
from IPython.display import Image, display
for p in glob.glob("/content/cphd_out/*/compare/compare.png"):
    print(p); display(Image(p))

In [ ]:
import glob, json
for p in glob.glob("/content/cphd_out/*/compare/report.json"):
    r = json.load(open(p))
    print(json.dumps({"test_A": r["test_A"], "test_B": r["test_B"]}, indent=2))

## Notes

* **Section 3 is the one that must pass.** It needs no data and validates the
  algorithm itself.
* **ω-K on a spotlight SICD** will give a poor round-trip number. That is the
  model mismatch, not a bug: ω-K assumes stripmap, straight track, no squint.
  Use inverse PFA (section 6) for spotlight data.
* **Test B coherence below 1 is expected** even when Test A passes — the real
  CPHD contains energy from outside the image and effects PFA discarded.
* Colab free tier: ~12 GB RAM and a disk that resets. Keep `--chip-size` at
  512–2048 and delete large downloads between runs.